# Topic 13 — Principal Component Analysis (PCA)

> **Goal of this notebook:** understand how PCA finds the directions of greatest variance to compress many features into a few, how to read **explained variance**, and reproduce it from scratch with an eigen-decomposition.

**Contents**
1. Concept & intuition
2. The mathematics (covariance, eigenvectors, projection)
3. When to use & assumptions
4. The dataset: Breast Cancer (30 features)
5. Explained variance & the scree plot
6. Visualising data in 2 principal components
7. PCA from scratch (eigen-decomposition)
8. PCA as preprocessing for a classifier
9. Pros, cons & when to use
10. Summary

## 1. Concept & Intuition

Many datasets have dozens of features that are **correlated** and partly redundant. PCA finds a new set of axes — the **principal components** — that are:
- **orthogonal** (uncorrelated), and
- ordered so the **first captures the most variance**, the second the next most, and so on.

By keeping only the first few components we describe most of the data with far fewer numbers — useful for **visualisation**, **compression**, **noise reduction**, and speeding up downstream models. PCA is an *unsupervised* linear transformation; it ignores labels and just follows the spread of the data.

## 2. The Mathematics

1. **Standardise** the features (mean 0, variance 1) — PCA is variance-based, so scale matters.
2. Compute the **covariance matrix** $\mathbf{C} = \frac{1}{m-1}\mathbf{X}^T\mathbf{X}$.
3. **Eigen-decompose** it: $\mathbf{C}\mathbf{v} = \lambda \mathbf{v}$. Each eigenvector $\mathbf{v}$ is a principal-component direction; its eigenvalue $\lambda$ is the variance along it.
4. Sort components by eigenvalue (descending) and keep the top $k$.
5. **Project** the data onto them: $\mathbf{Z} = \mathbf{X}\mathbf{V}_k$.

The **explained variance ratio** of component $j$ is $\lambda_j / \sum_i \lambda_i$. (In practice PCA is computed via the **SVD** of $\mathbf{X}$, which is numerically equivalent and more stable.)

## 3. When to Use & Assumptions

- Use when features are **correlated** and you want fewer dimensions with minimal information loss.
- Assumes the interesting structure lies in **high-variance, linear** directions.
- Components are linear combinations of original features → **less interpretable**.
- **Always scale first**; PCA is sensitive to feature units.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

np.random.seed(42)
plt.rcParams['figure.figsize'] = (8, 5)
print('Libraries loaded.')

## 4. The Dataset: Breast Cancer (30 Features)

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
X_s = StandardScaler().fit_transform(X)
print('Original dimensionality:', X_s.shape[1], 'features')

## 5. Explained Variance & the Scree Plot

How many components do we actually need? The cumulative explained variance tells us.

In [ ]:
pca = PCA().fit(X_s)
evr = pca.explained_variance_ratio_
cum = np.cumsum(evr)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].bar(range(1, len(evr) + 1), evr)
ax[0].set_xlabel('component'); ax[0].set_ylabel('explained variance ratio')
ax[0].set_title('Scree plot')
ax[1].plot(range(1, len(cum) + 1), cum, marker='o')
ax[1].axhline(0.95, color='r', ls='--', label='95%')
ax[1].set_xlabel('number of components'); ax[1].set_ylabel('cumulative variance')
ax[1].set_title('Cumulative explained variance'); ax[1].legend()
plt.show()

k95 = int(np.argmax(cum >= 0.95) + 1)
print(f'Components needed for 95% variance: {k95} (of {X_s.shape[1]})')
print('First 2 components explain:', round(cum[1] * 100, 1), '% of variance')

## 6. Visualising Data in 2 Principal Components

Projecting 30 features down to 2 still separates the classes well.

In [ ]:
Z = PCA(n_components=2, random_state=42).fit_transform(X_s)
plt.scatter(Z[:, 0], Z[:, 1], c=y, cmap='coolwarm', edgecolor='k', s=20)
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.title('Breast Cancer projected onto 2 principal components')
plt.colorbar(label='class'); plt.show()

## 7. PCA From Scratch (Eigen-decomposition)

Covariance → eigenvectors → project, then compare the explained-variance ratios to scikit-learn.

In [ ]:
# 1. covariance matrix of standardized data
C = np.cov(X_s, rowvar=False)
# 2. eigen-decomposition (symmetric -> use eigh)
eigvals, eigvecs = np.linalg.eigh(C)
# 3. sort descending
order = np.argsort(eigvals)[::-1]
eigvals, eigvecs = eigvals[order], eigvecs[:, order]
# 4. explained variance ratio
evr_scratch = eigvals / eigvals.sum()

print('Scratch  EVR (first 5):', np.round(evr_scratch[:5], 4))
print('sklearn  EVR (first 5):', np.round(pca.explained_variance_ratio_[:5], 4))
print('Match:', np.allclose(evr_scratch[:5], pca.explained_variance_ratio_[:5]))

## 8. PCA as Preprocessing for a Classifier

Does compressing to the 95%-variance components keep predictive power while cutting features?

In [ ]:
full = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
reduced = make_pipeline(StandardScaler(), PCA(n_components=k95),
                        LogisticRegression(max_iter=5000))

print(f'All 30 features      CV accuracy: {cross_val_score(full, X, y, cv=5).mean():.4f}')
print(f'{k95} PCA components  CV accuracy: {cross_val_score(reduced, X, y, cv=5).mean():.4f}')

## 9. Pros, Cons & When to Use

**Pros**
- Reduces dimensionality while retaining most variance (compression, speed, denoising).
- Removes correlation/multicollinearity; great for 2D visualisation.
- Fast and deterministic.

**Cons**
- Components are **linear** combinations → hard to interpret.
- Captures variance, not necessarily the directions useful for a *label* (it's unsupervised).
- Sensitive to scaling; struggles with strongly non-linear structure (use t-SNE/UMAP/kernel-PCA there).

**When to use**
- Preprocessing before distance-based or slow models.
- Visualising high-dimensional data.
- Compressing correlated features.

## 10. Summary

- PCA rotates the data onto **orthogonal axes of maximum variance** (eigenvectors of the covariance matrix).
- The **explained-variance ratio** / scree plot tells you how many components to keep.
- On Breast Cancer, a handful of components captured 95% of the variance and 2 components already separated the classes; the reduced model kept nearly the full accuracy.
- Our from-scratch eigen-decomposition reproduced scikit-learn's explained-variance ratios exactly.